# Optimization

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import json

In [2]:
inputs = dict(MAC = 0.4, # Mean Aerodynamic Chord [m]
AR = 14, # [-]
S = 5.5, # [m^2]
taper = 0.4, # [-]
rootchord = 0.5, # [m]
thicknessChordRatio = 0.17, # [-]
xAC = 0.25, # [-] position of ac with respect to the chord
mtom = 1972, # maximum take-off mass from statistical data - Class I estimation
S1 = 5.5,
S2 = 5.5, # surface areas of wing one and two
A = 14, # aspect ratio of a wing, not aircraft
nmax = 3.2, # maximum load factor
Pmax = 15.25, # this is defined as maximum perimeter in Roskam, so i took top down view of the fuselage perimeter
lf = 7.2, # length of fuselage
m_pax = 95, # average mass of a passenger according to Google
n_prop = 16, # number of engines
n_pax = 5, # number of passengers (pilot included)
pos_fus = 3.6, # fuselage centre of mass away from the nose
pos_lgear = 3.6, # landing gear position away from the nose
pos_frontwing = 0.2,
pos_backwing = 7, # positions of the wings away from the nose
m_prop = [30]*16, # list of mass of engines (so 30 kg per engine with nacelle and propeller)
pos_prop = [0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0], # 8 on front wing and 8 on back wing
Mac = 0.0645 * 0.5 * 1.2 * 53 ** 2 * 5.25 * 0.65, # aerodynamic moment around AC
flighttime = 3, # [hr]
turnovertime = 2, # we dont actually need this xd
takeofftime = 0.1,
enginePlacement = list(np.linspace(0.1*(14 * 5.5) ** 0.5/2, 0.8*(14 * 5.5) ** 0.5/2, 4)),
engineMass = 400 * 9.81 / 8,
Thover = 2960,
Tcruise = 2960,
p_pax = [1.5, 3, 3, 4.2, 4.2],
battery_pos = 3.6,
cargo_m = 85, cargo_pos = 6, battery_m = 400,
materialdata = '../data/materials.csv') | {json.loads(open('ldists.json', 'r').read())}

In [44]:
from Geometry import HatStringer, JStringer, ZStringer, WingBox, WingStructure
from SolveLoads import WingLoads, Engines, Fatigue
from Weight import *
from Material import Material


class StructuralError(Exception):
    def __init__(self, msg):
        super().__init__(msg)


class Structure:
    def __init__(self, **inputs):
        self.__dict__.update(inputs)
        base, height = 0.75 - 0.15, 0.11571117509557907 + 0.03145738125495376 # x/c, y/c
        self.n_ult = self.nmax*1.5
        self.hatGeom = dict(bflange1 = 0.02, bflange2 =0.02, tflange = 0.02, vflange = 0.035, tstr = 0.001)
        self.normalBox = WingBox(self.thicknessOfSkin, self.thicknessOfSpar, base, height)
        self.normalBox.StrPlacement(self.nStrT, self.nStrB, stringerGeometry = self.hatGeom, stringerType = 'Hat')
        self.span = (self.AR * self.S1) ** 0.5
        self.wing = WingStructure(self.span, self.taper, self.rootchord, self.normalBox)
        self.omax, self.taumax, self.Ymax, self.cycles, self.matsk, self.matstr, self.loads, self.critbuckling = [None]*8
        self.omin, self.taumin, self.Ymin = [None]*3
        self.wingWeight = None
        self.dragd, self.liftd, self.pos = np.array(self.dragd), np.array(self.liftd), np.array(self.pos)

    def __setitem__(self, key, item):
        self.__dict__[key] = item

    __getitem__ = lambda self, key: self.__dict__[key]
        
    def compute_stresses(self, nStrT, nStrB, thicknessOfSkin, thicknessOfSpar):
        self.dragd, self.liftd = np.array(self.dragd), np.array(self.liftd)
        args = dict(span=self.span, taper=self.taper, cr=self.rootchord, tsk=thicknessOfSkin,
                    tsp=thicknessOfSpar, toc=self.thicknessChordRatio, nStrT=nStrT, nStrB=nStrB,
                    strType='Hat', strGeo=self.hatGeom, mac=self.MAC, xac=self.xAC,
                    engines=Engines(self.Thover, self.Tcruise, self.enginePlacement,self.engineMass), frac=0.6)

        self.loads = WingLoads(**args)
        wing = Wing(self.mtom, self.S1, self.S2, self.n_ult, self.A, [self.pos_frontwing, self.pos_backwing])
        self.wingWeight = wingWeight = wing.mass[0]
        fuselage = Fuselage(self.mtom, self.Pmax, self.lf, self.n_pax, self.pos_fus)
        lgear = LandingGear(self.mtom, self.pos_lgear)
        props = Propulsion(self.n_prop, self.m_prop, self.pos_prop)

        w = Weight(self.m_pax, wing, fuselage, lgear, props,
                   cargo_m = self.cargo_m, cargo_pos = self.cargo_pos, battery_m = self.battery_m,
                   battery_pos = self.battery_pos, p_pax = self.p_pax)
        reactionsCruise = self.loads.equilibriumCruise([self.pos, self.dragd], [self.pos, self.liftd],
                                                  [self.pos, [self.Mac / self.span]*len(self.pos)], self.wingWeight)
        reactionsVTO = self.loads.equilibriumVTO(wingWeight)
        lift, wgt = self.loads.internalLoads([self.pos, self.dragd], [self.pos, self.liftd],
                                        [self.pos, [self.Mac / self.span]*len(self.pos)], wingWeight)
        VxVTO, MyVTO = self.loads.internalLoadsVTO(wingWeight)
        coords, o_cr, tau_cr, Ymcr = self.loads.stressesCruise()
        coords, o_VTO, tau_VTO, YmVTO = self.loads.stressesVTO()

        (self.omin, self.omax), (self.taumin, self.taumax) = WingLoads.extreme(coords, o_cr), WingLoads.extreme(coords, tau_cr)
        self.Ymin, self.Ymax = WingLoads.extreme(coords, Ymcr)
        return [[self.omin, self.omax], [self.taumin, self.taumax], [self.Ymin, self.Ymax]]

    def compute_fatigue(self):
        wingWeight = self.wingWeight
        liftdist = np.array(self.liftd) * self.n_ult
        dragdist = np.array(self.dragd) * self.n_ult
        pos, liftd, dragd, Mac, span, wingWeight = [self[k] for k in 'pos, liftd, dragd, Mac, span, wingWeight'.split(', ')]
        
        fatigue_reactionsCruise = self.loads.equilibriumCruise([pos, dragd], [pos, liftd], [pos, [Mac / span]*len(pos)], wingWeight)
        fatigue_lift, fatigue_wgt = self.loads.internalLoads([pos, dragd], [pos, liftd], [pos, [Mac / span]*len(pos)], wingWeight)
        coords, ocrf, taucrf, Ymcrf = self.loads.stressesCruise()

        fatigue_reactionsVTO = self.loads.equilibriumVTO(wingWeight)
        fatigue_VxVTO, fatigue_MyVTO = self.loads.internalLoadsVTO(wingWeight)
        coords, oVTOf, tauVTOf, YmVTOf = self.loads.stressesVTO()

        fatigue_reactionsVTOgr = self.loads.equilibriumVTO(wingWeight, ground = True)
        fatigue_VxVTOgr, fatigue_MyVTOgr = self.loads.internalLoadsVTO(wingWeight, ground = True)
        coords, oVTOfgr, tauVTOfgr, YmVTOfgr = self.loads.stressesVTO()

        *coor, maxDif = self.loads.extreme(coords, oVTOf - ocrf)
        ind = [i for i in range(len(coords)) if np.all(coords[i] == coor)][0]

        oVTOfgrmd, oVTOfmd, ocrfmd = oVTOfgr[ind], oVTOf[ind], ocrf[ind]

        fatigue = Fatigue(oVTOfgrmd, oVTOfmd, ocrfmd, flighttime, turnovertime, takeofftime, material)

        t, y = fatigue.determineCycle()
        fdf = fatigue.getCycles()
        self.cycles = fatigue.MinersRule()
        return self.cycles
    
    def compute_buckling(self, stringerMat, skinMat):

        root = self.loads.wing(0)
        EofStringers = self.matstr.E
        vOfStringers = self.matsk.v
        yieldOfStringers = self.matstr.oy
        EofSkin = self.matsk.E
        vOfSkin = self.matsk.v
        self.critbuckling = root.Bstress(EofStringers, vOfStringers, yieldOfStringers, EofSkin, vOfSkin)
        return self.critbuckling
    
    def optimize(self, stringerMat = dict(material='Al 6061', Condition='T6'),
                skinMat = dict(material='Al 6061', Condition='T6')):
        
        self.matsk = materialsk = Material.load(**(skinMat | {'file': self.materialdata}))
        self.matstr = materialstr = Material.load(**(stringerMat | {'file': self.materialdata}))

        nStrT, nStrB, thicknessOfSkin, thicknessOfSpar = \
        [self[k] for k in 'nStrT, nStrB, thicknessOfSkin, thicknessOfSpar'.split(', ')]
    
        while True:
            (omin, omax), (taumin, taumax), (Ymin, Ymax) = \
            self.compute_stresses(nStrT, nStrB, thicknessOfSkin, thicknessOfSpar) 
            root = self.loads.wing(0)

            critbuckling = self.compute_buckling(materialstr, materialsk)
            cycles = self.compute_fatigue()

            yieldMargin, bucklingmargin = matsk['oyield'] / Ymax[1], -critbuckling / omin[1]

            if omin[1] <= -critbuckling or Ymax[1] >= materialsk['oyield']:
                if bucklingmargin < yieldMargin:
                    if abs(Ymax[0][0] - root.b/2) < 1e-7 or abs(Ymax[0][0] + root.b/2) < 1e-7:
                        thicknessOfSpar += 0.0001
                    elif abs(Ymax[0][1] - root.h/2) < 1e-7 or abs(Ymax[0][1] + root.h/2) < 1e-7:
                        thicknessOfSkin += 0.0001
                    else:
                        raise StructuralError(f"Coordinates not on wingbox: {Ymax}")
                    nStrB += 1
                else:
                    nStrT += 1
                         
            elif 1 < bucklingmargin < 1.1 and 1 < yieldMargin < 1.1:
                break
            else:
                if bucklingmargin < yieldMargin:
                    nStrT -= 1
                else:
                    if abs(Ymax[0][0] - root.b/2) < 1e-7 or abs(Ymax[0][0] + root.b/2) < 1e-7:
                        thicknessOfSpar += 0.0001
                    elif abs(Ymax[0][1] - root.h/2) < 1e-7 or abs(Ymax[0][1] + root.h/2) < 1e-7:
                        thicknessOfSkin += 0.0001
                    nStrB -= 1
        
        self.nStrT, self.nStrB, self.thicknessOfSkin, self.thicknessOfSpar = nStrT, nStrB, thicknessOfSkin, thicknessOfSpar
        return nStrT, nStrB, thicknessOfSkin, thicknessOfSpar
        

In [45]:
struct = Structure(**(inputs | dict(nStrT=1, nStrB=1, thicknessOfSkin=1e-3, thicknessOfSpar=5e-3) ))
struct.optimize()

/home/ec2-user/DSE/structures/Equilibrium.py:47: IntegrationWarning: The maximum number of subdivisions (50) has been achieved.
  If increasing the limit yields no improvement it is advised to analyze 
  the integrand in order to determine the difficulties.  If the position of a 
  local difficulty can be determined (singularity, discontinuity) one will 
  probably gain from splitting up the interval and calling the integrator 
  on the subranges.  Perhaps a special-purpose integrator should be used.
  forces = [quad(lambda x: np.interp(x, self.p, self.v[i, :]), self.p[0], self.p[-1])[0] for i in range(2)]
/home/ec2-user/DSE/structures/Equilibrium.py:53: IntegrationWarning: The maximum number of subdivisions (50) has been achieved.
  If increasing the limit yields no improvement it is advised to analyze 
  the integrand in order to determine the difficulties.  If the position of a 
  local difficulty can be determined (singularity, discontinuity) one will 
  probably gain from splittin

IndexError: list index out of range